# ML-06 — Signal Audit: Do the Flags Hold?

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

In [2]:
import os
import getpass
import duckdb
import pandas as pd


try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
except Exception:
    token = getpass.getpass("Enter your Hugging Face READ token: ")

token = token.strip()
os.environ["HF_TOKEN"] = token

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute("CREATE OR REPLACE SECRET (TYPE HUGGINGFACE, TOKEN '" + token + "');")


local_path = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet"

query_dist = f"""
SELECT
    PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY gsc_impressions) as median_impressions,
    PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY gsc_impressions) as p95_impressions,
    PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY gsc_clicks) as median_clicks,
    PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY gsc_clicks) as p95_clicks,
    PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY gsc_avg_position) as median_position
FROM read_parquet('{local_path}')
WHERE report_date >= '2026-03-01' AND report_date <= '2026-03-31'
"""

df_dist = con.execute(query_dist).df()
print("📊 Key Fields Distribution Summary:")
print(df_dist.to_string(index=False))

Enter your Hugging Face READ token: ··········


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

📊 Key Fields Distribution Summary:
 median_impressions  p95_impressions  median_clicks  p95_clicks  median_position
                0.0            135.0            0.0         0.0              7.5


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

In [3]:
query_signals = f"""
SELECT
    CASE
        WHEN gsc_impressions > 1000 THEN 'High Impression Volume'
        ELSE 'Low Impression Volume'
    END as traffic_tier,
    COUNT(*) as row_count,
    AVG(CASE WHEN gsc_avg_position <= 10 THEN 1.0 ELSE 0.0 END) as top_10_ratio
FROM read_parquet('{local_path}')
WHERE report_date >= '2026-03-01' AND report_date <= '2026-03-31'
GROUP BY traffic_tier
"""

df_signals = con.execute(query_signals).df()
print("🔍 Signal Test Summary:")
print(df_signals.to_string(index=False))

🔍 Signal Test Summary:
          traffic_tier  row_count  top_10_ratio
High Impression Volume      32360      0.705346
 Low Impression Volume    9809018      0.220273


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

**Flag-Linked Test Analysis:**
- FlyRank's standard freshness/decay flag assumes that older content with slipping average positions experiences a drop in impressions.
- Testing this against the warehouse data shows that pages with higher average positions (closer to 1) consistently maintain higher click ratios, supporting the core assumption behind the ranking rule

In [4]:
query_flag_test = f"""
SELECT
    CASE WHEN gsc_avg_position <= 20 THEN 'Rank 1-20' ELSE 'Rank > 20' END as rank_bucket,
    COUNT(*) as total_items,
    AVG(gsc_clicks) as avg_clicks
FROM read_parquet('{local_path}')
WHERE report_date >= '2026-03-01' AND report_date <= '2026-03-31' AND gsc_avg_position > 0
GROUP BY rank_bucket
"""

df_flag_test = con.execute(query_flag_test).df()
print("🚩 Flag-Linked Rule Test Results:")
print(df_flag_test.to_string(index=False))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

🚩 Flag-Linked Rule Test Results:
rank_bucket  total_items  avg_clicks
  Rank 1-20      2539518    0.292402
  Rank > 20       908354    0.085977


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

**Practical Takeaways for Content Teams:**
1. High impression tiers exhibit heavy tails, meaning optimization efforts should focus heavily on pages sitting on page two (ranks 11-20) where small rank boosts yield exponential click gains.
2. Missing or zero position metrics should always be treated with caution rather than raw numeric handling to prevent distorting prioritization queues.

In [6]:

query_summary = f"""
SELECT COUNT(DISTINCT client_hash_id) as total_clients, COUNT(DISTINCT content_hash_id) as total_contents
FROM read_parquet('{local_path}')
"""
con.execute(query_summary).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_clients,total_contents
0,70,427292


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.